In [2]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime

In [3]:
# каталог с ноутбуками и корень проекта
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from src.analytics import assess, guess_mapping, normalize, read_table
from src.analytics.quality import drop_duplicates, trim_incomplete_tail
from src.analytics.schema import AMOUNT, CUSTOMER_ID, ORDER_DATE
# данные
DATA_PATH = ROOT / "data" / "raw" / "online_retail_II.xlsx"
raw = read_table(DATA_PATH)
source = "Online Retail II"

raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
# распознавание колонок
mapping = guess_mapping(raw)

for canonical, column in mapping.as_dict().items():
    print(f"{canonical:<12} <- {column}")

if mapping.unmapped:
    print("\nНе распознано:", ", ".join(map(str, mapping.unmapped)))

assert mapping.is_valid, f"Не хватает колонок: {mapping.missing_required}"

order_date   <- InvoiceDate
customer_id  <- Customer ID
order_id     <- Invoice
quantity     <- Quantity
unit_price   <- Price

Не распознано: StockCode, Description, Country


In [5]:
# отчистка 
clean = normalize(raw, mapping)
clean, duplicates = drop_duplicates(clean)
clean, tail_trimmed = trim_incomplete_tail(clean)
clean.head()

,order_date,customer_id,amount,order_id,quantity,unit_price
0,2009-12-01 07:45:00,13085,83.4,489434,12.0,6.95
1,2009-12-01 07:45:00,13085,81.0,489434,12.0,6.75
2,2009-12-01 07:45:00,13085,100.8,489434,48.0,2.10
3,2009-12-01 07:45:00,13085,30.0,489434,24.0,1.25
4,2009-12-01 07:45:00,13085,39.6,489434,24.0,1.65


In [7]:
df = clean.copy()
snapshot_data = df['order_date'].max()
rfm_user = df.groupby('customer_id').agg(
    Recency=('order_date', lambda x: (snapshot_data - x.max()).days),
    Frequency=('order_id', 'nunique'),
    Monetary=('amount', 'sum')
).reset_index()

rfm_user['R'] = pd.qcut(rfm_user['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_user['F'] = pd.qcut(rfm_user['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_user['M'] = pd.qcut(rfm_user['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Функция для объединения оценок в текстовые сегменты
def assign_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    # Крупный чек: тратят очень много, независимо от частоты, но не "Потеряные"
    if m == 5 and r > 1 and f < 4:
        return 'Крупный чек'
    # Чемпионы: покупали недавно, часто и много
    elif r >= 4 and f >= 4 and m >= 4:
        return 'Чемпионы'
    # Лояльные: покупают относительно часто, стабильно
    elif r >= 3 and f >= 3:
        return 'Лояльные'
    # Новички: купили впервые или совсем недавно, мало заказов
    elif r >= 4 and f == 1:
        return 'Новички'
    # Под угрозой: покупали раньше часто/много, но давно не возвращались
    elif r <= 2 and (f >= 3 or m >= 4):
        return 'Под угрозой'
    # Спящие: давние клиенты с редкими или средними покупками
    elif r == 3 and f <= 2:
        return 'Спящие'
    # Потерянные: покупали давно, мало и редко
    return 'Потерянные'

rfm_user['Segment'] = rfm_user.apply(assign_segment, axis=1)

# Считаем глобальные показатели по КЛИЕНТСКОЙ базе
total_customers = rfm_user['customer_id'].nunique()
total_revenue = rfm_user['Monetary'].sum()

# Финальная группировка по сегментам
rfm_summary = rfm_user.groupby('Segment').agg(
    Количество_клиентов=('customer_id', 'count'), # Теперь считает уникальных людей!
    Выручка=('Monetary', 'sum'),
    Средний_чек=('Monetary', 'mean')
).reset_index()

# Расчет долей
rfm_summary['Доля_базы_%'] = round((rfm_summary['Количество_клиентов'] / total_customers) * 100, 2)
rfm_summary['Доля_выручки_%'] = round((rfm_summary['Выручка'] / total_revenue) * 100, 2)
rfm_summary['Средний_чек'] = round(rfm_summary['Средний_чек'], 2)

# Колонка рекомендаций
strategies = {
    'Чемпионы': 'Награждать, вовлекать в VIP-программы, предлагать эксклюзивные новинки.',
    'Лояльные': 'Предлагать сопутствующие товары (Cross-sell), просить отзывы, давать бонусы.',
    'Новички': 'Проводить онбординг, слать welcome-цепочки писем, дать скидку на 2-й заказ.',
    'Под угрозой': 'Активировать: слать персональные скидки, проводить опросы "Что случилось?".',
    'Спящие': 'Прогревать контентом, напоминать о себе, предлагать легкие подарки к заказу.',
    'Потерянные': 'Включать в автоматические реактивационные рассылки. При отсутствии реакции — удалять.',
    'Крупный чек': 'Предлагать премиальные товары (Up-sell), не спамить мелкими скидками.'
}

rfm_summary['Что делать'] = rfm_summary['Segment'].map(strategies)
# Настройка красивого вывода больших чисел (без e+08)
pd.options.display.float_format = '{:.2f}'.format

# Сортируем по выручке, чтобы сразу увидеть ключевые сегменты
rfm_summary = rfm_summary.sort_values(by='Выручка', ascending=False)
print(rfm_summary.to_string(index=False))

    Segment  Количество_клиентов    Выручка  Средний_чек  Доля_базы_%  Доля_выручки_%                                                                            Что делать
   Чемпионы                  952 4111249.83      4318.54        21.72           68.11               Награждать, вовлекать в VIP-программы, предлагать эксклюзивные новинки.
   Лояльные                  937  788673.36       841.70        21.38           13.07          Предлагать сопутствующие товары (Cross-sell), просить отзывы, давать бонусы.
Под угрозой                  740  667883.68       902.55        16.88           11.06           Активировать: слать персональные скидки, проводить опросы "Что случилось?".
 Потерянные                 1216  207649.30       170.76        27.74            3.44 Включать в автоматические реактивационные рассылки. При отсутствии реакции — удалять.
Крупный чек                   57  134939.72      2367.36         1.30            2.24                 Предлагать премиальные товары (Up-sell